# Dimensionality Reduction

Sometimes the "actual" dimensions of the data might not correspond to the dimensions we have. 
 
**PCA** finds the directions in your data where the points vary the most. 

The steps are the following: 

**1)** Find the direction with maximum variance → first principal component 

**2)** Project the data onto that direction 

**3)** Remove that direction from the data (the explained part) 

**4)** Repeat to find the next directions 

<img src="https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/pca_directions_1.png" width="400">
<img src="https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/pca_directions_2.png" width="400">

**Before using PCA** 

**-** Scale Features before applying PCA

**-** Accuracy might drop

## Steps to build the model

In [ ]:
# Download the helper modules used in this lab (run once)
import urllib.request

BASE = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/"
for f in ["linear_algebra.py", "gradient_descent.py"]:
    urllib.request.urlretrieve(BASE + f, f)

**1. Centre the data:** 
We translate the data so that every dimension has mean 0:

In [ ]:
from linear_algebra import subtract,vector_mean,Vector
from typing import List, Dict

def de_mean(data: List[Vector]) -> List[Vector]:
    """Recenters the data to have mean 0 in every dimension"""
    mean = vector_mean(data)
    return [subtract(vector, mean) for vector in data]

**2. Find the direction that captures the highest variance:** 
Given a direction d ( a vector of magnitude 1), each row x in the matrix extends dot(x,d) in the d direction. And every non zero vector w determines a direction if we rescale it to have magnitude 1.
first principal component.

In [ ]:
from linear_algebra import magnitude

def direction(w: Vector) -> Vector:
    mag = magnitude(w)
    return [w_i / mag for w_i in w] # Rescale the vector so it has magnitude 1 (We only care about the direction, not the magnitude)

**3. Compute the variance in the direction w:**  
Given a non zero vector w, we can compute the variance of the dataset in the direction determined by w

In [ ]:
from linear_algebra import dot

def directional_variance(data: List[Vector], w: Vector) -> float:
    """
    Returns the variance of x in the direction of w
    """
    w_dir = direction(w) # rescale to get a vector of magnitude 1
    return sum(dot(v, w_dir) ** 2 for v in data) #total variance of the entire dataset along w direction

**4. Compute the gradient of that variance** 
We need to find the direction vector that maximizes this variance. To do so, let's find the derivate of the variance with respect to w

In [ ]:
def directional_variance_gradient(data: List[Vector], w: Vector) -> Vector:
    """
    The gradient of directional variance with respect to w
    """
    w_dir = direction(w)
    return [sum(2 * dot(v, w_dir) * v[i] for v in data)
            for i in range(len(w))] # The result is a list with the gradient for each element of the directional vector

**5. Iterate until convergence (gradient descent)** 
Let's use the gradient descent to find the directional vector w that maximizes the variance. This vector would be the First Principal Component

In [ ]:
import tqdm
from gradient_descent import gradient_step

def first_principal_component(data: List[Vector],
                              n: int = 100,
                              step_size: float = 0.1) -> Vector:
    # Start with a random guess
    guess = [1.0 for _ in data[0]]

    with tqdm.trange(n) as t:
        for _ in t:
            dv = directional_variance(data, guess)
            gradient = directional_variance_gradient(data, guess)
            guess = gradient_step(guess, gradient, step_size)
            t.set_description(f"dv: {dv:.3f}")

    return direction(guess)

**6. Project the data onto the direction vector** 


In [ ]:
from linear_algebra import scalar_multiply

def project(v: Vector, w: Vector) -> Vector:
    """return the projection of v onto the direction w"""
    projection_length = dot(v, w) #scalar (number) length of the projection if w is a unit vector.
    return scalar_multiply(projection_length, w) # The result is a new vector with the
                                                 #projection vector or data onto the directional vector ( we need to do this for each data in the dataset)


**7. Remove the first component from the data**
If we need to find further components, we first remove the projections from the data because this new component must capture new variance, not variance already explained by w₁. The result is the part of v that is not aligned with w.

In [ ]:
from linear_algebra import subtract

def remove_projection_from_vector(v: Vector, w: Vector) -> Vector:
    """projects v onto w and subtracts the result from v"""
    return subtract(v, project(v, w))# Removes the part of v that lies in the direction w


Let's do the same for the whole dataset:

In [ ]:
def remove_projection(data: List[Vector], w: Vector) -> List[Vector]:
    return [remove_projection_from_vector(v, w) for v in data]

**8. Iterate to find as many components as you need**
We can find the next principal component, that would be the direction of maximum remaining variance, by repeating the process on the result of the remove-projection

In [ ]:
def pca(data: List[Vector], num_components: int) -> List[Vector]:
    components: List[Vector] = []
    for _ in range(num_components):
        component = first_principal_component(data) # Find the direction vector that explains the highest variance
        components.append(component) # Save the vestor in a list of components
        data = remove_projection(data, component) # Remove the projection of the data onto this directional vector

    return components

**9. Re-express your data in terms of the new axes**
We project every point onto the principal components: this is the data in the reduced PCA space, ready for analysis or regression.

In [ ]:
def transform_vector(v: Vector, components: List[Vector]) -> Vector:
    return [dot(v, w) for w in components]

Let's do this for the whole dataset:

In [ ]:
def transform(data: List[Vector], components: List[Vector]) -> List[Vector]:
    return [transform_vector(v, components) for v in data]

### Trying it on a toy dataset

Let's use our functions on 100 points where `X_2` grows with `X_1` (like the figure at the top), centre them, and ask for the two principal components.

In [ ]:
import random
import matplotlib.pyplot as plt
from linear_algebra import vector_mean

random.seed(0)
points = []
for _ in range(100):
    x1 = random.uniform(10, 40)
    points.append([x1, 3 * x1 + random.gauss(0, 15)])

centred = de_mean(points)                 # step 1
components = pca(centred, 2)              # steps 2-8
print("Components:", [[round(c, 3) for c in w] for w in components])

# Variance captured along each component (per point)
for k, w in enumerate(components, start=1):
    print(f"PC{k}: variance = {directional_variance(centred, w) / len(centred):.1f}")

# Plot the centred data with the first component
plt.scatter([v[0] for v in centred], [v[1] for v in centred], s=15)
w1 = components[0]
plt.plot([-40 * w1[0], 40 * w1[0]], [-40 * w1[1], 40 * w1[1]], color="firebrick")
plt.axis("equal"); plt.xlabel("X_1 (centred)"); plt.ylabel("X_2 (centred)")
plt.show()

new_axes = transform(centred, components)   # step 9
print("First point in the new axes:", [round(z, 2) for z in new_axes[0]])

The first component (red line) follows the direction in which the points are most spread out, and almost all of the variance is on PC1. Notice that `X_2` varies much more than `X_1` simply because of its **scale**: this is why we standardise the features before PCA when they have different units.

## PCA using sklearn
Let's see an example when trying to predict a number based on the pixels of the image: 

<img src="https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/digits_1.png" width="400">
<img src="https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%207%20-%20PCA/digits_2.png" width="400">


In [ ]:
from sklearn.datasets import load_digits
import pandas as pd

dataset = load_digits()
dataset.keys()

In [ ]:
dataset.data.shape

In [ ]:
dataset.data[0]

In [ ]:
dataset.data[0].reshape(8,8) # For visualization

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline
plt.gray()
plt.matshow(dataset.data[0].reshape(8,8))

In [ ]:
dataset.target[:5]

In [ ]:
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df.head()

In [ ]:
dataset.target

In [ ]:
X = df
y = dataset.target

Let's standardize using the functions from sklearn:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

Let's divide the sample in 80%-20%

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=30)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
model.score(X_test, y_test)

## Use PCA to reduce dimensions

In [ ]:
from sklearn.decomposition import PCA

pca_sk = PCA(0.95) # Create new features keeping the 95% of the variance of the data
X_pca = pca_sk.fit_transform(X)
X_pca.shape

The number of features changed from **64** to **29**. It doesn't remove features: it creates new ones that capture the variance of the original ones.

In [ ]:
pca_sk.explained_variance_ratio_

**The explained variance ratio** indicates how much of the variance is captured by each component

In [ ]:
pca_sk.n_components_


In [ ]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=30)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_pca, y_train)
model.score(X_test_pca, y_test)

By reducing the original components, it's also reduced the accuracy from **97,2%** to **96,9%**

In [ ]:
pca_sk = PCA(n_components=2)
X_pca = pca_sk.fit_transform(X)
X_pca.shape
X_pca

In [ ]:
pca_sk.explained_variance_ratio_

With 2 components, we are only capturing **28%** of the variance of the original data

In [ ]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=30)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_pca, y_train)
model.score(X_test_pca, y_test)

With only 2 components, the accuracy of the model dropped to **60%**

### Classifying Flowers

The Iris Dataset is a well known database in the pattern recognition literature. The data set contains 3 classes of 50 instances each, where each class refers to a type of iris plant.

Four features are given:
1. sepal length in cm 
2. sepal width in cm 
3. petal length in cm 
4. petal width in cm 

We will explore this dataset next.

In [ ]:
from sklearn import datasets

iris = datasets.load_iris()
X = iris.data
y = iris.target

print(X[:10])
print(y[:10])

list(iris.target_names)

Your job is to create a model that predicts the target variable, and then reduce the features to 3 and see how much of the accuracy changes with the reduction.
